# Atlas tests v2: stronger learner, matched bridge test, random-features PoE, equal bytes

**How to run:** Runtime → Change runtime type → **T4 GPU** → Runtime → **Run all**. Expect about 1.5 to 2 hours. Keep the tab open.
At the end `atlas_v2b_results.json` downloads. Send it back. If any cell fails, paste the error.

**Why v2 exists (from the v1 results):**
- v1's learner was too weak. Its head got 8.1% vs ewt's 17.4%, and its learned features with perfect means (25.1%) lost to fixed random features (35.1%).
  - So v2 first **calibrates the learner** on a separate seed and picks it using held-out control items, never the test set.
- v1 compared a 10-per-class reserve against a ~20-per-class buffer. That mixed up *how many* items with *whether they were trained on*.
  - v2 adds a **matched-size** test with the same count per class from each source.
- v1 never combined the random-features expert with the learned one. v2 adds the **product of experts (PoE)** and the **ε-mixture**, and checks the mixture's log-loss bound on every test item.
- v1's "equal bytes" config spent half the memory on a reserve. v2's config **E** uses 1800 buffer + 2 control items per class = 2000 images, like ewt's ER baseline.

**Honesty notes**
- These are fresh runs of my re-implementation, not ewt's T-S5 runs. Comparisons inside the notebook are clean; absolute numbers are not comparable to ewt's.
- The random chart's exact statistics are not free. A 1024×1024 covariance plus 100 means is about 4.6 MB in float32, which is roughly 1500 CIFAR images' worth. So config E is equal in **images**, not in total bytes. This is reported, not hidden.
- Every learned arm uses the same LDA covariance within a config. Temperatures, PoE weights and the learner choice are all fitted on **control** items. The test set is used only for the final numbers.

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   # before CUDA starts
import sys, json, time, gc, importlib, subprocess, platform
import numpy as np, torch
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)
import warnings; warnings.filterwarnings('once')
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('torch', torch.__version__, '| numpy', np.__version__, '| python', platform.python_version(), '| GPU', GPU)
if GPU is None:
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU. On CPU this takes many hours.')
VERSION = 'atlas-tests-v2b'

## Pre-registration (written before any v2 result)

**Learner calibration** (seed 100, which is not one of the test seeds). Six candidates:
- v1's recipe;
- momentum;
- several gradient steps per incoming batch;
- flip + crop augmentation (training only);
- a small learning-rate range.

The pick is the candidate with the highest **learned-feature ceiling on control items** (LDA with true means). The gate "learned ceiling > random-chart LDA on control" is reported. If the gate fails, the learned-arm rankings are provisional.

**Configs** (5 seeds each, 20 tasks × 5 classes, single online pass):
- **A**: buffer 2000 + reserve 20/class (10 bridge, 10 control). Σ comes from reserve items.
- **E**: buffer 1800 + 2 control/class = 2000 images. No bridge. Σ comes from the buffer plus control.

**Arms.**
- `O_full`: oracle means.
- `B_only_buf`: buffer means.
- `CH_cv_buf`: X2 global map.
- `GR_prev_buf`: per-class GREG on the previous chart.
- `GR_multi_buf`: GREG with [r, previous].
- `GR_stable_buf`: one hop from r.
- Matched pairs (A only), with the same k per class at every hop: `GR_prev_res_m` vs `GR_prev_buf_m`, and `B_only_res_m` vs `B_only_buf_m`.
- `RC_LDA`: the random chart with exact statistics.
- `PoE[RC+x]`: w·log p_RC + (1−w)·log p_x, with w and temperatures fitted on control.
- `Mix[RC,PoE+x]`: 0.05·p_RC + 0.95·p_PoE.

**Comparisons (rule: |mean| > 2·SE and 5/5 seeds same sign; not significant ≠ tie). My predictions, with confidence:**

| | Comparison | Prediction |
|---|---|---|
| G1 | A: O_full > RC_LDA | yes, 60% |
| P1 | A: GR_prev_buf > CH_cv_buf | yes, 80% |
| P2 | A: GR_multi_buf > GR_prev_buf | yes, 40% |
| P3 | A: GR_prev_res_m > GR_prev_buf_m (training bias is real) | yes, 35% |
| P3b | A: B_only_res_m > B_only_buf_m | yes, 40% |
| P4 | A: PoE[RC+GR_prev_buf] > RC_LDA | yes, 80% |
| P4b | A: PoE[RC+GR_prev_buf] > GR_prev_buf | yes, 85% |
| P5 | E: PoE[RC+GR_prev_buf] > CH_cv_buf | yes, 85% |
| P5b | E: PoE[RC+GR_prev_buf] > RC_LDA | yes, 70% |
| P6 | A: CH_cv_buf > B_only_buf (X2 replication) | yes, 40% |
| P7 | A: PoE[RC+O_full] > PoE[RC+GR_prev_buf] (migration loss remains) | yes, 70% |

**Hard check:** the ε-mixture bound −log p ≤ −log p_RC − log ε must hold on 100% of test items. It is a mathematical identity, so any violation means a code bug.

**Edge cases and what happens**

| What if | Handling |
|---|---|
| A learner candidate diverges (NaN loss) | It is marked failed in calibration and cannot be picked. |
| The picked learner diverges on a test seed | The run is recorded as an error and excluded. Comparisons use only the seeds every arm has. |
| The learner still loses to random features | The gate is printed. The learned rankings are labelled provisional; the PoE results still stand. |
| Tuning leaks into the test set | It can't: the learner is picked on seed 100 using control items; temperatures and PoE weights use control items; test is read once. |
| A class has no items in the buffer at some hop | It falls back to the global map, and the fallback is counted and reported. |
| The matched test isn't really matched | Each class gets the same k from each source at every hop by construction; the mean k is printed. |
| Colab disconnects | Calibration and every run are saved as they finish. Run all again and finished work is skipped. |
| A results file from another version | It is detected by the version and learner settings and not mixed in. |
| Many comparisons | There are 11. Under no effect, a 5/5 same-sign result has probability 1/16 per test, before the 2·SE requirement. Expect at most one false positive. Look at sizes, not just verdicts. |
| Augmentation contaminates features | Augmentation is training only. All features are extracted from clean images. |

## 1. Code (pure-numpy analysis; PyTorch learner)

In [ ]:
%%writefile atlas_lib.py
# ===== atlas_lib: pure-numpy analysis code (estimators, LDA, statistics) =====
import numpy as np

def ledoit_wolf(Xc):
    """Ledoit-Wolf shrinkage covariance of already-centred data Xc (n x p).
    Same formula as sklearn.covariance.ledoit_wolf (assume_centered=True)."""
    Xc = np.asarray(Xc, dtype=np.float64)
    n, p = Xc.shape
    X2 = Xc ** 2
    emp_cov_trace = X2.sum(0) / n
    mu = emp_cov_trace.sum() / p
    beta_ = (X2.T @ X2).sum()
    delta_ = ((Xc.T @ Xc) ** 2).sum() / n ** 2
    beta = 1.0 / (p * n) * (beta_ / n - delta_)
    delta = (delta_ - 2.0 * mu * emp_cov_trace.sum() + p * mu ** 2) / p
    beta = min(beta, delta)
    s = 0.0 if beta == 0 else beta / delta
    S = Xc.T @ Xc / n
    return (1 - s) * S + s * mu * np.eye(p), s

def class_center(X, y):
    """Subtract each sample's class mean. Returns centred X and dict of class means."""
    Xc = np.empty_like(X, dtype=np.float64)
    means = {}
    for c in np.unique(y):
        m = y == c
        means[c] = X[m].mean(0)
        Xc[m] = X[m] - means[c]
    return Xc, means

def pooled_cov(X, y):
    Xc, _ = class_center(X, y)
    return ledoit_wolf(Xc)[0]

def lda_predict(X, M, Sigma):
    """Uniform-prior LDA. M: (C x d) class means (rows), Sigma: (d x d). Returns argmax index into rows of M."""
    P = np.linalg.solve(Sigma, M.T)            # d x C  = Sigma^-1 mu_c
    b = -0.5 * np.einsum('dc,cd->c', P, M)     # -1/2 mu^T Sigma^-1 mu
    S = X @ P + b
    return S.argmax(1)

EIGH = np.linalg.eigh   # the notebook may swap in a GPU float64 eigh with identical results (checked in a cell)

def _ridge_factory(Xs, Y):
    """Returns a function lam -> ridge coefficients for standardised Xs, using the smaller Gram matrix."""
    n, p = Xs.shape
    if n >= p:
        ev, V = EIGH(Xs.T @ Xs); ev = np.clip(ev, 0, None)
        VtXtY = V.T @ (Xs.T @ Y)
        return lambda lam: V @ (VtXtY / (ev + lam)[:, None])
    ev, U = EIGH(Xs @ Xs.T); ev = np.clip(ev, 0, None)
    UtY = U.T @ Y
    return lambda lam: Xs.T @ (U @ (UtY / (ev + lam)[:, None]))

def ridge_cv(X, Y, rng, k=5, ngrid=16, split=None, kappas=(0.0, 0.1, 0.3, 1.0)):
    """Ridge regression Y ~ X @ B on centred data (no intercept), penalty chosen by k-fold CV on (X,Y) only.
    Columns of X are standardised internally; returned B is in original units.
    split=p1: columns [:p1] form a separate block (the stable chart r) whose penalty is chosen separately,
    by scaling that block by kappa (kappa=0 drops it). This stops a wide noisy block from forcing one
    shared penalty onto the informative block."""
    X = np.asarray(X, np.float64); Y = np.asarray(Y, np.float64)
    n = X.shape[0]
    sd = X.std(0) + 1e-8
    Xs0 = X / sd
    ks = kappas if split else (1.0,)
    k = max(2, min(k, n))
    folds = rng.permutation(n) % k
    best = (np.inf, None, None)
    for kap in ks:
        scale = np.ones(X.shape[1])
        if split: scale[:split] = kap
        Xs = Xs0 * scale if not (split and kap == 0) else Xs0[:, split:]
        smax2 = np.linalg.norm(Xs, 2) ** 2 + 1e-12
        lams = smax2 * np.logspace(-9, 1, ngrid)
        err = np.zeros(ngrid)
        for f in range(k):
            tr, te = folds != f, folds == f
            coef = _ridge_factory(Xs[tr], Y[tr])
            for jj, lam in enumerate(lams):
                err[jj] += ((Xs[te] @ coef(lam) - Y[te]) ** 2).sum()
        jj = int(err.argmin())
        if err[jj] < best[0]:
            best = (err[jj], kap, lams[jj])
    _, kap, lam = best
    if split and kap == 0:
        B = np.zeros((X.shape[1], Y.shape[1]))
        B[split:] = _ridge_factory(Xs0[:, split:], Y)(lam) / sd[split:, None]
        return B, lam, kap
    scale = np.ones(X.shape[1])
    if split: scale[:split] = kap
    B = _ridge_factory(Xs0 * scale, Y)(lam) * (scale / sd)[:, None]
    return B, lam, kap

def greg_means(classes, mu_aux, A_br, Y_br, y_br, rng, crossfit=False, split=None):
    """Regression (control-variate / GREG) estimate of class means of Y.
    classes : list of class ids to estimate
    mu_aux  : dict class -> population mean of auxiliary features (known or carried)
    A_br, Y_br, y_br : bridge items' auxiliary features, target features, labels
    Estimate: mu_Y,c = mean_B,c(Y) + (mu_aux,c - mean_B,c(A)) @ B, B = pooled within-class slope.
    Default: one pooled slope fitted on all bridge items (crossfit=False). Optional 2-fold cross-fitting;
    on synthetic data it lost badly with small bridges (5 items/class), so it is off (decided before any CIFAR run).
    Classes with no bridge items fall back to a global affine map (flagged).
    Returns dict class -> estimate, set of fallback classes, info."""
    A_br = np.asarray(A_br, np.float64); Y_br = np.asarray(Y_br, np.float64)
    fold = np.zeros(len(y_br), int)
    for c in np.unique(y_br):
        idx = np.where(y_br == c)[0]
        fold[idx[rng.permutation(len(idx))]] = np.arange(len(idx)) % 2
    out, fallback = {}, set()
    present = set(np.unique(y_br).tolist())
    if crossfit:
        Bs = []
        for f in (0, 1):
            tr = fold != f
            Ac, _ = class_center(A_br[tr], y_br[tr]); Yc, _ = class_center(Y_br[tr], y_br[tr])
            Bs.append(ridge_cv(Ac, Yc, rng, split=split)[0])
    Ac_all, _ = class_center(A_br, y_br); Yc_all, _ = class_center(Y_br, y_br)
    B_all = ridge_cv(Ac_all, Yc_all, rng, split=split)[0]
    for c in classes:
        if c not in present:
            fallback.add(c)
            out[c] = Y_br.mean(0) + (mu_aux[c] - A_br.mean(0)) @ B_all
            continue
        if not crossfit:
            m = y_br == c
            out[c] = Y_br[m].mean(0) + (mu_aux[c] - A_br[m].mean(0)) @ B_all
            continue
        est, w = 0.0, 0
        for f in (0, 1):
            m = (y_br == c) & (fold == f)
            if m.sum() == 0:
                continue
            est = est + m.sum() * (Y_br[m].mean(0) + (mu_aux[c] - A_br[m].mean(0)) @ Bs[1 - f])
            w += m.sum()
        out[c] = est / w
    return out, fallback

def global_affine_means(classes, mu_aux, A_br, Y_br, y_br, rng, split=None):
    """X2-style chained map: pooled within-class-centred ridge slope, global intercept."""
    A_br = np.asarray(A_br, np.float64); Y_br = np.asarray(Y_br, np.float64)
    Ac, _ = class_center(A_br, y_br); Yc, _ = class_center(Y_br, y_br)
    B = ridge_cv(Ac, Yc, rng, split=split)[0]
    a0, y0 = A_br.mean(0), Y_br.mean(0)
    return {c: y0 + (mu_aux[c] - a0) @ B for c in classes}

def within_class_r2(A, Y, y, rng, split=None):
    """Out-of-fold within-class R^2 of Y predicted from A (2-fold cross-fit, centring by training-fold class means)."""
    A = np.asarray(A, np.float64); Y = np.asarray(Y, np.float64)
    fold = np.zeros(len(y), int)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        fold[idx[rng.permutation(len(idx))]] = np.arange(len(idx)) % 2
    sse = sst = 0.0
    for f in (0, 1):
        tr, te = fold != f, fold == f
        Ac, ma = class_center(A[tr], y[tr]); Yc, my = class_center(Y[tr], y[tr])
        B = ridge_cv(Ac, Yc, rng, split=split)[0]
        for c in np.unique(y[te]):
            if c not in ma:
                continue
            m = te & (y == c)
            pred = my[c] + (A[m] - ma[c]) @ B
            sse += ((Y[m] - pred) ** 2).sum()
            sst += ((Y[m] - my[c]) ** 2).sum()
    return 1 - sse / sst

def paired(a, b):
    """Pre-registered rule: diff = a-b over seeds; significant iff |mean| > 2*SE and all seeds share the sign."""
    d = np.asarray(a, float) - np.asarray(b, float)
    n = len(d)
    se = d.std(ddof=1) / np.sqrt(n) if n > 1 else float('nan')
    sig = bool(n > 1 and abs(d.mean()) > 2 * se and (np.all(d > 0) or np.all(d < 0)))
    return dict(mean=float(d.mean()), se=float(se), n_pos=int((d > 0).sum()), n=n, sig=sig)

def spearman(a, b):
    ra = np.argsort(np.argsort(a)); rb = np.argsort(np.argsort(b))
    return float(np.corrcoef(ra, rb)[0, 1])

# ---------- v2 additions: scores, calibration, product of experts, epsilon-mixture ----------
def lda_scores(X, M, Sigma):
    """Uniform-prior LDA discriminant scores (n x C); argmax equals lda_predict."""
    P = np.linalg.solve(Sigma, M.T)
    return X @ P - 0.5 * np.einsum('dc,cd->c', P, M)

def log_softmax(S):
    S = S - S.max(1, keepdims=True)
    return S - np.log(np.exp(S).sum(1, keepdims=True))

TAUS = np.exp(np.linspace(np.log(0.05), np.log(50), 41))

def fit_temperature(S, yi):
    """Temperature tau minimising NLL of softmax(S/tau) on (S, yi). Grid search (robust, no optimiser state)."""
    nll = [-log_softmax(S / t)[np.arange(len(yi)), yi].mean() for t in TAUS]
    return float(TAUS[int(np.argmin(nll))])

def fit_poe(L1, L2, yi, grid=np.linspace(0, 1, 21)):
    """Weight w for w*L1 + (1-w)*L2 (log-probs of two calibrated experts), chosen on control data by accuracy,
    ties broken by NLL after temperature calibration. Returns w, tau of the combination."""
    best = None
    for w in grid:
        Z = w * L1 + (1 - w) * L2
        acc = (Z.argmax(1) == yi).mean()
        tau = fit_temperature(Z, yi)
        nll = -log_softmax(Z / tau)[np.arange(len(yi)), yi].mean()
        key = (acc, -nll)
        if best is None or key > best[0]:
            best = (key, float(w), tau)
    return best[1], best[2]

def eps_mixture_logp(LA, LP, eps):
    """log( eps*pA + (1-eps)*pP ) computed stably from log-probs."""
    a = np.log(eps) + LA; b = np.log1p(-eps) + LP
    m = np.maximum(a, b)
    return m + np.log(np.exp(a - m) + np.exp(b - m))

In [ ]:
%%writefile atlas_analysis.py
# ===== atlas_analysis v2: everything after the features exist (pure numpy) =====
import numpy as np, time
from atlas_lib import (pooled_cov, lda_scores, log_softmax, fit_temperature, fit_poe, eps_mixture_logp,
                       greg_means, global_affine_means, within_class_r2)

EPS = 0.05          # epsilon of the stable-expert mixture, fixed in advance
MATCH_CAP = 10      # items per class in the matched-size bridge test

def run_analysis(D, seed):
    """D: dict with
         tasks, y_train, stream_ids, bridge_ids (may be empty), control_ids (>= 2 per class)
         F[t]   features of ALL train items under snapshot f_t (end of task t);  R  stable-chart features of all train items
         buffers[t]  train ids in the replay buffer at the end of task t
         Ftest, Rtest, y_test, head_acc
         sigma_from  'reserve' (bridge+control items) or 'buffer' (final buffer + control): the ONE Sigma all arms share
    """
    rng = np.random.default_rng(10_000 + seed)
    t0 = time.time()
    tasks = [np.asarray(c) for c in D['tasks']]; T = len(tasks)
    classes = np.concatenate(tasks); C = len(classes)
    cidx = {int(c): i for i, c in enumerate(classes)}
    y = np.asarray(D['y_train']); F = D['F']; R = D['R']
    stream = np.asarray(D['stream_ids'])
    br = np.asarray(D['bridge_ids'], dtype=np.int64); ct = np.asarray(D['control_ids'], dtype=np.int64)
    has_bridge = len(br) > 0
    # ---- integrity asserts (leakage, coverage) ----
    res_ids = np.concatenate([br, ct])
    assert len(np.intersect1d(stream, res_ids)) == 0, 'reserve leaked into stream'
    assert len(np.intersect1d(br, ct)) == 0, 'bridge and control overlap'
    for b in D['buffers']:
        assert np.isin(b, stream).all(), 'buffer holds a non-stream item'
    assert set(np.unique(y[ct]).tolist()) == set(classes.tolist()), 'control must cover every class'
    if has_bridge:
        assert set(np.unique(y[br]).tolist()) == set(classes.tolist()), 'bridge must cover every class'
    f64 = lambda a: np.asarray(a, np.float64)
    task_of = {int(c): t for t, cs in enumerate(tasks) for c in cs}
    ids_of = {int(c): stream[y[stream] == c] for c in classes}
    FT = F[T - 1]
    final_buf = np.asarray(D['buffers'][T - 1])
    # ---- exact statistics ----
    mu_r = {c: f64(R[ids]).mean(0) for c, ids in ids_of.items()}
    mu_rec = {c: f64(F[task_of[c]][ids]).mean(0) for c, ids in ids_of.items()}
    oracle = {c: f64(FT[ids]).mean(0) for c, ids in ids_of.items()}
    sig_ids = res_ids if D['sigma_from'] == 'reserve' else np.concatenate([final_buf, ct])
    Sig = pooled_cov(f64(FT[sig_ids]), y[sig_ids])
    Sig_r = pooled_cov(f64(R[stream]), y[stream])          # stable chart keeps exact sufficient statistics
    ytest = np.asarray(D['y_test']); keep = np.isin(ytest, classes)
    Ftest = f64(D['Ftest'])[keep]; Rtest = f64(D['Rtest'])[keep]; ytest = ytest[keep]
    yti = np.array([cidx[int(c)] for c in ytest]); yci = np.array([cidx[int(c)] for c in y[ct]])
    old_mask = np.isin(ytest, np.concatenate(tasks[: T // 2]))
    Fct, Rct = f64(FT[ct]), f64(R[ct])
    Mr = np.stack([mu_r[int(c)] for c in classes])

    # ---- arms (class means in f_T) ----
    arms, fallbacks = {'O_full': oracle}, {}

    def bonly(pool):
        g = f64(FT[pool]).mean(0); out = {}; nfb = 0
        for c in classes:
            m = pool[y[pool] == c]
            if len(m) == 0: out[int(c)] = g; nfb += 1
            else: out[int(c)] = f64(FT[m]).mean(0)
        return out, nfb
    arms['B_only_buf'], fallbacks['B_only_buf'] = bonly(final_buf)

    def chain(kind, bridges):
        """bridges[t] = item ids used for the hop f_{t-1} -> f_t (t = 1..T-1)"""
        est = {int(c): mu_rec[int(c)] for c in tasks[0]}; nfb = 0
        for t in range(T - 1):
            old = [int(c) for c in np.concatenate(tasks[: t + 1])]
            ids = bridges[t + 1]
            A = f64(F[t][ids]); Y = f64(F[t + 1][ids]); yb = y[ids]
            if kind == 'multi':
                A = np.hstack([f64(R[ids]), A]); mu_aux = {c: np.concatenate([mu_r[c], est[c]]) for c in old}
                est, fb = greg_means(old, mu_aux, A, Y, yb, rng, split=R.shape[1]); nfb += len(fb)
            elif kind == 'aff':
                est = global_affine_means(old, est, A, Y, yb, rng)
            else:
                est, fb = greg_means(old, est, A, Y, yb, rng); nfb += len(fb)
            for c in tasks[t + 1]:
                est[int(c)] = mu_rec[int(c)]
        return est, nfb

    buf_bridges = [np.asarray(b) for b in D['buffers']]
    arms['CH_cv_buf'], fallbacks['CH_cv_buf'] = chain('aff', buf_bridges)
    arms['GR_prev_buf'], fallbacks['GR_prev_buf'] = chain('prev', buf_bridges)
    arms['GR_multi_buf'], fallbacks['GR_multi_buf'] = chain('multi', buf_bridges)
    est, fb = greg_means([int(c) for c in classes], mu_r, f64(R[final_buf]), f64(FT[final_buf]), y[final_buf], rng)
    arms['GR_stable_buf'], fallbacks['GR_stable_buf'] = est, len(fb)

    matched_n = None
    if has_bridge:
        # Matched-size bridge test: at every hop, per class, the SAME number k_c = min(cap, #buffer, #reserve) of items
        # from the buffer (trained on) and from the reserve (never trained on). Only the provenance differs.
        mrng = np.random.default_rng(20_000 + seed)
        mb, mr = [None], [None]; ks = []
        for t in range(1, T):
            seen = np.concatenate(tasks[: t + 1]); bb = np.asarray(D['buffers'][t]); sb, sr = [], []
            for c in seen:
                pb = bb[y[bb] == c]; pr = br[y[br] == c]
                k = min(MATCH_CAP, len(pb), len(pr))
                if k == 0: continue
                sb.append(mrng.choice(pb, k, replace=False)); sr.append(mrng.choice(pr, k, replace=False)); ks.append(k)
            mb.append(np.concatenate(sb)); mr.append(np.concatenate(sr))
        matched_n = float(np.mean(ks))
        arms['GR_prev_buf_m'], fallbacks['GR_prev_buf_m'] = chain('prev', mb)
        arms['GR_prev_res_m'], fallbacks['GR_prev_res_m'] = chain('prev', mr)
        arms['B_only_buf_m'], fallbacks['B_only_buf_m'] = bonly(mb[-1])
        arms['B_only_res_m'], fallbacks['B_only_res_m'] = bonly(mr[-1])

    # ---- scoring: one Sigma for all learned arms; experts calibrated and combined on CONTROL items only ----
    out = {'acc': {}, 'acc_old': {}, 'acc_recent': {}, 'nll': {}, 'fallbacks': fallbacks, 'poe_w': {}, 'matched_n': matched_n}
    def record(name, Ltest):
        pred = Ltest.argmax(1)
        out['acc'][name] = float((pred == yti).mean() * 100)
        out['acc_old'][name] = float((pred[old_mask] == yti[old_mask]).mean() * 100)
        out['acc_recent'][name] = float((pred[~old_mask] == yti[~old_mask]).mean() * 100)
        out['nll'][name] = float(-Ltest[np.arange(len(yti)), yti].mean())
    tau_r = fit_temperature(lda_scores(Rct, Mr, Sig_r), yci)
    L_rc_ct = log_softmax(lda_scores(Rct, Mr, Sig_r) / tau_r); L_rc_te = log_softmax(lda_scores(Rtest, Mr, Sig_r) / tau_r)
    record('RC_LDA', L_rc_te)
    viol = {}
    for name, means in arms.items():
        M = np.stack([means[int(c)] for c in classes])
        S_ct, S_te = lda_scores(Fct, M, Sig), lda_scores(Ftest, M, Sig)
        tau = fit_temperature(S_ct, yci)
        L_ct, L_te = log_softmax(S_ct / tau), log_softmax(S_te / tau)
        record(name, L_te)
        if name in ('O_full', 'B_only_buf', 'CH_cv_buf', 'GR_prev_buf', 'GR_multi_buf'):
            w, tau_p = fit_poe(L_rc_ct, L_ct, yci)
            Lp = log_softmax((w * L_rc_te + (1 - w) * L_te) / tau_p)
            record(f'PoE[RC+{name}]', Lp); out['poe_w'][name] = w
            Lm = eps_mixture_logp(L_rc_te, Lp, EPS)
            record(f'Mix[RC,PoE+{name}]', Lm)
            idx = np.arange(len(yti))
            viol[name] = int((-Lm[idx, yti] > -L_rc_te[idx, yti] - np.log(EPS) + 1e-9).sum())
    out['mixture_bound_violations'] = viol
    out['acc']['ER_head'] = float(D['head_acc'])
    # ---- Test 1: within-class R^2 on held-out items (only with >= 10 held-out items per class) ----
    r2 = {}
    if len(res_ids) >= 10 * C:
        Yr = f64(FT[res_ids]); yr = y[res_ids]
        r2['fT|r'] = within_class_r2(f64(R[res_ids]), Yr, yr, rng)
        if T >= 2:
            r2['fT|f(T-1)'] = within_class_r2(f64(F[T - 2][res_ids]), Yr, yr, rng)
            r2['fT|[r,f(T-1)]'] = within_class_r2(np.hstack([f64(R[res_ids]), f64(F[T - 2][res_ids])]), Yr, yr, rng, split=R.shape[1])
            r2['fT|f1'] = within_class_r2(f64(F[0][res_ids]), Yr, yr, rng)
            hops = [within_class_r2(f64(F[t][res_ids]), f64(F[t + 1][res_ids]), yr, rng) for t in range(T - 1)]
            r2['hop_mean'] = float(np.mean(hops)); r2['hop_min'] = float(np.min(hops))
    out['r2'] = {k: float(v) for k, v in r2.items()}
    out['seconds'] = time.time() - t0
    return out

In [ ]:
%%writefile atlas_summary.py
# ===== Summary and pre-registered comparisons — v2 =====
import numpy as np
from atlas_lib import paired

HYPOTHESES = [
    ('G1 stronger learner beats random features (oracle means)',        ('A', 'O_full'), ('A', 'RC_LDA')),
    ('P1 per-class GREG beats global map (v1: +4.3)',                    ('A', 'GR_prev_buf'), ('A', 'CH_cv_buf')),
    ('P2 adding r as auxiliary helps',                                    ('A', 'GR_multi_buf'), ('A', 'GR_prev_buf')),
    ('P3 matched size: reserve bridge beats buffer bridge (= training bias is real)', ('A', 'GR_prev_res_m'), ('A', 'GR_prev_buf_m')),
    ('P3b matched size, raw means: reserve beats buffer',                 ('A', 'B_only_res_m'), ('A', 'B_only_buf_m')),
    ('P4 PoE beats random features alone',                                ('A', 'PoE[RC+GR_prev_buf]'), ('A', 'RC_LDA')),
    ('P4b PoE beats migrated learned readout alone',                      ('A', 'PoE[RC+GR_prev_buf]'), ('A', 'GR_prev_buf')),
    ('P5 equal bytes: PoE beats X2-style global map',                     ('E', 'PoE[RC+GR_prev_buf]'), ('E', 'CH_cv_buf')),
    ('P5b equal bytes: PoE beats random features alone',                  ('E', 'PoE[RC+GR_prev_buf]'), ('E', 'RC_LDA')),
    ('P6 X2 replication: global map beats buffer-only',                   ('A', 'CH_cv_buf'), ('A', 'B_only_buf')),
    ('P7 migration loss left inside the PoE (oracle means vs migrated)',  ('A', 'PoE[RC+O_full]'), ('A', 'PoE[RC+GR_prev_buf]')),
]

def table(results, cfg, metric):
    runs = results[cfg]; seeds = sorted(s for s in runs if 'error' not in runs[s])
    arms = list(runs[seeds[0]][metric].keys())
    rows = []
    for a in arms:
        v = np.array([runs[s][metric][a] for s in seeds])
        rows.append((a, float(v.mean()), float(v.std(ddof=1) / np.sqrt(len(v))) if len(v) > 1 else float('nan')))
    rev = metric != 'nll'
    return sorted(rows, key=lambda r: -r[1] if rev else r[1]), seeds

def summarize(results, print_fn=print):
    out = {'tables': {}, 'hypotheses': [], 'r2': {}, 'checks': {}}
    for cfg in sorted(results):
        bad = [s for s in results[cfg] if 'error' in results[cfg][s]]
        if bad: print_fn(f'\nConfig {cfg}: runs that failed and are excluded: {bad}')
        for metric in ('acc', 'acc_old', 'acc_recent', 'nll'):
            rows, seeds = table(results, cfg, metric)
            out['tables'][f'{cfg}/{metric}'] = rows
            print_fn(f'\nConfig {cfg}, {metric}  (mean ± SE over seeds {seeds})')
            for a, m, s in rows: print_fn(f'   {a:28s} {m:8.3f} ± {s:.3f}')
        seeds = sorted(s for s in results[cfg] if 'error' not in results[cfg][s])
        viol = sum(sum(results[cfg][s]['mixture_bound_violations'].values()) for s in seeds)
        out['checks'][f'{cfg}/mixture_bound_violations'] = int(viol)
        print_fn(f'\nConfig {cfg}: ε-mixture bound violations on test items (must be 0): {viol}')
        print_fn(f'Config {cfg}: PoE weight on random-chart expert, per arm: ' +
                 str({a: round(float(np.mean([results[cfg][s]['poe_w'][a] for s in seeds])), 2) for a in results[cfg][seeds[0]]['poe_w']}))
        mn = [results[cfg][s]['matched_n'] for s in seeds if results[cfg][s]['matched_n'] is not None]
        if mn: print_fn(f'Config {cfg}: matched bridge items per class (mean): {np.mean(mn):.2f}')
        keys = list(results[cfg][seeds[0]]['r2'])
        if keys:
            out['r2'][cfg] = {k: (float(np.mean([results[cfg][s]['r2'][k] for s in seeds])),
                                  float(np.std([results[cfg][s]['r2'][k] for s in seeds], ddof=1) / np.sqrt(len(seeds))) if len(seeds) > 1 else float('nan'))
                              for k in keys}
            print_fn(f'Config {cfg}, within-class R² on held-out items:')
            for k, (m, s) in out['r2'][cfg].items(): print_fn(f'   {k:16s} {m:.3f} ± {s:.3f}')
    print_fn('\nPre-registered comparisons (test accuracy). Significant = |mean| > 2·SE and all seeds same sign. Not significant ≠ tie.')
    for name, (ca, aa), (cb, ab) in HYPOTHESES:
        if ca not in results or cb not in results: continue
        seeds = sorted(s for s in set(results[ca]) & set(results[cb]) if 'error' not in results[ca][s] and 'error' not in results[cb][s])
        if not seeds or aa not in results[ca][seeds[0]]['acc'] or ab not in results[cb][seeds[0]]['acc']: continue
        r = paired([results[ca][s]['acc'][aa] for s in seeds], [results[cb][s]['acc'][ab] for s in seeds])
        verdict = ('SUPPORTED' if r['mean'] > 0 else 'REVERSED') if r['sig'] else 'NOT SIGNIFICANT (not a tie)'
        out['hypotheses'].append(dict(name=name, a=f'{ca}/{aa}', b=f'{cb}/{ab}', result=r, verdict=verdict))
        print_fn(f'  {name}\n     {ca}/{aa} − {cb}/{ab}: {r["mean"]:+.2f} ± {r["se"]:.2f} ({r["n_pos"]}/{r["n"]} seeds positive) -> {verdict}')
    return out

In [ ]:
%%writefile atlas_torch.py
# ===== Learner, stable random chart, online stream, feature extraction (PyTorch) — v2 =====
import copy, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as Fn, torchvision
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
MEAN = torch.tensor([0.5071, 0.4865, 0.4409], device=DEV).view(1, 3, 1, 1)
STD = torch.tensor([0.2673, 0.2564, 0.2762], device=DEV).view(1, 3, 1, 1)

def load_cifar100(root='/content/data'):
    tr = torchvision.datasets.CIFAR100(root, train=True, download=True)
    te = torchvision.datasets.CIFAR100(root, train=False, download=True)
    to_t = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).contiguous().to(DEV)
    Xtr, ytr, Xte, yte = to_t(tr.data), np.array(tr.targets), to_t(te.data), np.array(te.targets)
    assert Xtr.shape == (50000, 3, 32, 32) and Xte.shape == (10000, 3, 32, 32)
    assert (np.bincount(ytr) == 500).all() and (np.bincount(yte) == 100).all()
    return Xtr, ytr, Xte, yte

def to_input(xu8):
    return (xu8.float() / 255.0 - MEAN) / STD

def augment(x, g):
    """Random horizontal flip + 4-pixel reflect-pad random crop on a FLOAT batch, driven by generator g."""
    n = x.shape[0]
    flip = torch.rand(n, generator=g, device=x.device) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    xp = Fn.pad(x, (4, 4, 4, 4), mode='reflect')
    off = torch.randint(0, 9, (n, 2), generator=g, device=x.device).tolist()
    return torch.stack([xp[i, :, a:a + 32, b:b + 32] for i, (a, b) in enumerate(off)])

def make_encoder():
    return nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1), nn.GroupNorm(8, 32), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 128, 3, padding=1), nn.GroupNorm(8, 128), nn.ReLU())

def embed(enc, x):
    return enc(x).mean((2, 3))

class Net(nn.Module):
    def __init__(self):
        super().__init__(); self.enc = make_encoder(); self.head = nn.Linear(128, 100)
    def forward(self, x):
        return self.head(embed(self.enc, x))

class RandomChart:
    """Stable reference r: fixed random 5x5 conv (256 maps) + ReLU + 2x2 average pool -> 1024-d. Never trained."""
    def __init__(self, seed, width=256):
        g = torch.Generator().manual_seed(seed)
        self.W = (torch.randn(width, 3, 5, 5, generator=g) / np.sqrt(75.0)).to(DEV)
        self.b = (0.1 * torch.randn(width, generator=g)).to(DEV)
    def __call__(self, x):
        return Fn.adaptive_avg_pool2d(Fn.relu(Fn.conv2d(x, self.W, self.b, padding=2)), 2).flatten(1)

@torch.no_grad()
def extract(fn, Xu8, bs=2500):
    return torch.cat([fn(to_input(Xu8[i:i + bs])).float().cpu() for i in range(0, len(Xu8), bs)]).numpy()

class Diverged(RuntimeError):
    pass

def split_data(seed, cfg, ytr):
    """Class order, held-out reserve (bridge, control) and stream. Depends only on the seed and sizes."""
    rng = np.random.default_rng(seed)
    order = rng.permutation(100)[: cfg['n_classes']]
    tasks = order.reshape(cfg['n_tasks'], -1)
    nb, nc = cfg['bridge_per_class'], cfg['control_per_class']
    bridge, control, stream_by_class = [], [], {}
    for c in order:
        ids = rng.permutation(np.where(ytr == c)[0])
        bridge.append(ids[:nb]); control.append(ids[nb:nb + nc])
        s = ids[nb + nc:]
        if cfg.get('stream_per_class'): s = s[: cfg['stream_per_class']]
        stream_by_class[int(c)] = s
    return rng, order, tasks, np.concatenate(bridge).astype(np.int64), np.concatenate(control).astype(np.int64), stream_by_class

def run_one(seed, cfg, data, log=print, full=True):
    """One online class-incremental run.
    full=True: returns the dict D for run_analysis.  full=False (calibration): returns summary numbers only."""
    Xtr, ytr, Xte, yte = data
    t0 = time.time()
    rng, order, tasks, bridge, control, stream_by_class = split_data(seed, cfg, ytr)
    stream_ids = np.concatenate([stream_by_class[int(c)] for c in order])
    assert len(np.intersect1d(stream_ids, np.concatenate([bridge, control]))) == 0
    torch.manual_seed(seed)
    g = torch.Generator(device=DEV).manual_seed(seed)
    model = Net().to(DEV)
    opt = torch.optim.SGD(model.parameters(), lr=cfg['lr'], momentum=cfg['momentum'])
    ytr_t = torch.from_numpy(ytr).to(DEV)
    M, B, iters, aug = cfg['M'], cfg['batch'], cfg['iters'], cfg['aug']
    buf = np.zeros(M, dtype=np.int64); nbuf = 0; nseen = 0
    buffers, snaps = [], []
    for t, cs in enumerate(tasks):
        ids = rng.permutation(np.concatenate([stream_by_class[int(c)] for c in cs]))
        for i in range(0, len(ids), B):
            bi = ids[i:i + B]
            for _ in range(iters):                      # several steps on the same incoming batch, fresh replay each step
                allidx = np.concatenate([bi, buf[rng.integers(0, nbuf, size=B)]]) if nbuf > 0 else bi
                it = torch.from_numpy(allidx).to(DEV)
                x = to_input(Xtr[it])
                if aug: x = augment(x, g)
                loss = Fn.cross_entropy(model(x), ytr_t[it])
                if not torch.isfinite(loss):
                    raise Diverged(f'non-finite loss at task {t}')
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            for j in bi:                                # reservoir sampling (Algorithm R), after the steps
                nseen += 1
                if nbuf < M: buf[nbuf] = j; nbuf += 1
                else:
                    k = rng.integers(0, nseen)
                    if k < M: buf[k] = j
        buffers.append(buf[:nbuf].copy())
        if full or t == len(tasks) - 1:
            snaps.append(copy.deepcopy(model.enc).eval())
    t_train = time.time() - t0
    model.eval()
    with torch.no_grad():
        def head_acc(X, yy):
            keep = np.isin(yy, order)
            lg = torch.cat([model(to_input(X[i:i + 2500])) for i in range(0, len(X), 2500)]).cpu().numpy()[keep]
            return float((order[lg[:, order].argmax(1)] == yy[keep]).mean() * 100)
        acc_test = head_acc(Xte, yte)
        ct_t = torch.from_numpy(control).to(DEV)
        acc_ctl = head_acc(Xtr[ct_t], ytr[control])
    if not full:   # calibration: learned-feature ceiling measured on CONTROL items only (never the test set)
        from atlas_lib import pooled_cov, lda_predict
        f = lambda X: extract(lambda x: embed(snaps[-1], x), X)
        Fs, Fc = f(Xtr[torch.from_numpy(stream_ids).to(DEV)]).astype(np.float64), f(Xtr[ct_t]).astype(np.float64)
        ys, yc = ytr[stream_ids], ytr[control]
        Mm = np.stack([Fs[ys == c].mean(0) for c in order])
        o_ctl = float((order[lda_predict(Fc, Mm, pooled_cov(Fs, ys))] == yc).mean() * 100)
        log(f'    head(control) {acc_ctl:.1f} | oracle-LDA(control) {o_ctl:.1f} | train {t_train:.0f}s')
        return dict(head_ctl=acc_ctl, oracle_ctl=o_ctl, seconds=t_train)
    F = [extract(lambda x, e=e: embed(e, x), Xtr) for e in snaps]
    chart = RandomChart(seed + 777)
    R, Rtest = extract(chart, Xtr, bs=500), extract(chart, Xte, bs=500)
    Ftest = extract(lambda x: embed(snaps[-1], x), Xte)
    for a in F + [R, Rtest, Ftest]:
        assert np.isfinite(a).all(), 'non-finite features'
    log(f'  seed {seed}: train {t_train:.0f}s, total {time.time()-t0:.0f}s, ER head acc {acc_test:.1f}, buffer {nbuf}, '
        f'per-class buffer min {np.bincount(ytr[buf[:nbuf]], minlength=100)[order].min()}')
    return dict(tasks=[np.array(c) for c in tasks], y_train=ytr, stream_ids=stream_ids, bridge_ids=bridge,
                control_ids=control, F=F, R=R, buffers=buffers, Ftest=Ftest, Rtest=Rtest, y_test=yte, head_acc=acc_test,
                sigma_from=cfg['sigma_from'])

def rc_control_acc(seed, cfg, data):
    """Random-chart LDA on CONTROL items (the bar the learner should clear). Same split as run_one."""
    from atlas_lib import pooled_cov, lda_predict
    Xtr, ytr, _, _ = data
    _, order, _, _, control, sbc = split_data(seed, cfg, ytr)
    stream_ids = np.concatenate([sbc[int(c)] for c in order])
    chart = RandomChart(seed + 777)
    Rs = extract(chart, Xtr[torch.from_numpy(stream_ids).to(DEV)], bs=500).astype(np.float64)
    Rc = extract(chart, Xtr[torch.from_numpy(control).to(DEV)], bs=500).astype(np.float64)
    ys, yc = ytr[stream_ids], ytr[control]
    Mm = np.stack([Rs[ys == c].mean(0) for c in order])
    return float((order[lda_predict(Rc, Mm, pooled_cov(Rs, ys))] == yc).mean() * 100)

In [ ]:
sys.path.insert(0, '.')
import atlas_lib, atlas_analysis, atlas_summary, atlas_torch
for m in (atlas_lib, atlas_analysis, atlas_summary, atlas_torch): importlib.reload(m)
from atlas_analysis import run_analysis
from atlas_summary import summarize
from atlas_torch import load_cifar100, run_one, rc_control_acc, Diverged

## 2. Checks with known answers (the notebook stops if anything fails)

In [ ]:
%%writefile test_lib.py
import numpy as np, sys
sys.path.insert(0,'.')
from atlas_lib import *
rng=np.random.default_rng(0)
fails=[]
def check(name, cond, info=''):
    print(('PASS' if cond else 'FAIL'), name, info); 
    if not cond: fails.append(name)

# L1 Ledoit-Wolf: shrinkage in [0,1]; identity data -> near mu*I; large n -> near sample cov
X=rng.normal(size=(5000,10)); C,s=ledoit_wolf(X-X.mean(0)); check('LW_large_n', np.abs(C-np.cov(X.T,bias=True)).max()<0.05 and 0<=s<=1, f's={s:.3f}')
X=rng.normal(size=(20,50)); C,s=ledoit_wolf(X-X.mean(0)); check('LW_p>n_wellcond', np.linalg.cond(C)<1e3 and s>0.3, f's={s:.3f}')

# L2 LDA equals brute-force Gaussian log-likelihood argmax
d,Cn=8,5; M=rng.normal(size=(Cn,d))*2; A=rng.normal(size=(d,d)); Sig=A@A.T+np.eye(d)
X=rng.normal(size=(300,d))
Si=np.linalg.inv(Sig); ll=np.stack([-0.5*np.einsum('nd,de,ne->n',X-M[c],Si,X-M[c]) for c in range(Cn)],1)
check('LDA_bruteforce', np.array_equal(lda_predict(X,M,Sig), ll.argmax(1)))

# L3 LDA accuracy on well separated Gaussians ~ Bayes
L=np.repeat(np.arange(Cn),400); Z=M[L]+rng.multivariate_normal(np.zeros(d),Sig,size=len(L))
acc=(lda_predict(Z,M,Sig)==L).mean(); check('LDA_acc_sane', acc>0.5, f'acc={acc:.3f}')

# L4 GREG exact under exact linear relation, no noise (truth recovered)
da,dy,Cn=6,4,10; W=rng.normal(size=(da,dy)); off=rng.normal(size=(Cn,dy))*0  # pure linear: y = a@W + b
b=rng.normal(size=dy)
mu_a={c:rng.normal(size=da)*3 for c in range(Cn)}
yb=np.repeat(np.arange(Cn),12); Ab=np.stack([mu_a[c] for c in yb])+rng.normal(size=(len(yb),da)); Yb=Ab@W+b
est,fb=greg_means(list(range(Cn)),mu_a,Ab,Yb,yb,rng)
err=max(np.abs(est[c]-(mu_a[c]@W+b)).max() for c in range(Cn)); check('GREG_exact_linear', err<1e-3, f'maxerr={err:.2e}')

# L5 GREG variance reduction ~ (1-R2) (Monte Carlo, scalar target, known population means)
res={}
for R2 in (0.0,0.8):
    errs_g=[];errs_b=[]
    for t in range(300):
        N=2000; a=rng.normal(size=(N,3)); e=rng.normal(size=N)
        w=np.array([1.,0.5,-0.5]); w=w/np.linalg.norm(w)*np.sqrt(R2)
        yv=(a@w+np.sqrt(1-R2)*e)[:,None]
        idx=rng.choice(N,30,replace=False)
        est,_=greg_means([0],{0:a.mean(0)},a[idx],yv[idx],np.zeros(30,int),rng)
        errs_g.append(est[0][0]-yv.mean()); errs_b.append(yv[idx].mean()-yv.mean())
    res[R2]=(np.var(errs_g)/np.var(errs_b))
check('GREG_varratio_R2=0', 0.8<res[0.0]<1.35, f'ratio={res[0.0]:.2f} (expect ~1)')
check('GREG_varratio_R2=0.8', res[0.8]<0.4, f'ratio={res[0.8]:.2f} (expect ~0.2 + slope noise)')

# L6 fallback path flagged for classes absent from bridge
est,fb=greg_means([0,1,99],{0:np.zeros(da),1:np.ones(da),99:np.ones(da)},Ab,Yb,yb,rng)
check('GREG_fallback_flag', fb=={99})

# L7 chained exact linear hops recover truth (two hops, invertible maps, exact)
mu0={c:rng.normal(size=5)*3 for c in range(6)}
W1=rng.normal(size=(5,5)); W2=rng.normal(size=(5,5))
yb=np.repeat(np.arange(6),10); F0=np.stack([mu0[c] for c in yb])+rng.normal(size=(60,5))
F1=F0@W1+1.0; F2=F1@W2-2.0
m1,_=greg_means(list(range(6)),mu0,F0,F1,yb,rng); m2,_=greg_means(list(range(6)),m1,F1,F2,yb,rng)
err=max(np.abs(m2[c]-((mu0[c]@W1+1)@W2-2)).max() for c in range(6)); check('chain_exact', err<1e-3, f'{err:.1e}')
g=global_affine_means(list(range(6)),mu0,F0,F1,yb,rng); err=max(np.abs(g[c]-(mu0[c]@W1+1)).max() for c in range(6)); check('global_affine_exact', err<1e-3, f'{err:.1e}')

# L8 within-class R2: ~1 for exact linear, ~0 for independent
A=rng.normal(size=(400,6)); yy=np.repeat(np.arange(20),20)
check('R2_linear', within_class_r2(A,A@rng.normal(size=(6,3)),yy,rng)>0.99)
r0=within_class_r2(A,rng.normal(size=(400,3)),yy,rng); check('R2_indep', abs(r0)<0.1, f'{r0:.3f}')

# L9 paired rule
check('paired_sig', paired([2,2.1,1.9,2.2,2.0],[0,0,0,0,0])['sig'] and not paired([1,-1,1,1,1],[0]*5)['sig'])
# L10 ridge equals closed form both regimes
for n,p in ((200,30),(30,200)):
    X=rng.normal(size=(n,p)); Y=rng.normal(size=(n,3))
    from atlas_lib import _ridge_factory
    lam=3.7; B=_ridge_factory(X,Y)(lam); Bc=np.linalg.solve(X.T@X+lam*np.eye(p),X.T@Y)
    check(f'ridge_closed_form_{n}x{p}', np.abs(B-Bc).max()<1e-8)
# L11 block ridge: kappa=0 path equals plain ridge on the second block; useless wide block is dropped or shrunk
Xa=rng.normal(size=(300,8)); Yb=Xa@rng.normal(size=(8,4))+0.1*rng.normal(size=(300,4))
Xn=rng.normal(size=(300,200))
Bb,lamb,kap=ridge_cv(np.hstack([Xn,Xa]),Yb,np.random.default_rng(1),split=200)
r2_block=1-((np.hstack([Xn,Xa])@Bb-Yb)**2).sum()/(Yb**2).sum()
check('block_ridge_drops_noise', kap<=0.1 and r2_block>0.95, f'kappa={kap} R2={r2_block:.3f}')
Bz=np.zeros((208,4)); 
B0,lam0,_=ridge_cv(Xa,Yb,np.random.default_rng(1))
check('block_ridge_shape', Bb.shape==(208,4))
print('FAILS:',fails)

import sys as _s
if fails: _s.exit(1)

In [ ]:
%%writefile test_lib_v2.py
import numpy as np, sys; sys.path.insert(0, '.')
from atlas_lib import *
rng = np.random.default_rng(5); fails = []
def check(n, c, i=''):
    print('PASS' if c else 'FAIL', n, i)
    if not c: fails.append(n)
d, C = 6, 5; M = rng.normal(size=(C, d)) * 2; A = rng.normal(size=(d, d)); Sig = A @ A.T + np.eye(d); X = rng.normal(size=(200, d))
check('scores_argmax_eq_predict', np.array_equal(lda_scores(X, M, Sig).argmax(1), lda_predict(X, M, Sig)))
L = log_softmax(rng.normal(size=(50, 7)) * 30)
check('log_softmax_normalised', np.allclose(np.exp(L).sum(1), 1) and np.isfinite(L).all())
# temperature: data generated from softmax(S/2) -> fitted tau near 2
S = rng.normal(size=(20000, 10)) * 3; P = np.exp(log_softmax(S / 2.0)); yi = np.array([rng.choice(10, p=p) for p in P])
t = fit_temperature(S, yi); check('temperature_recovered', 1.6 < t < 2.5, f'tau={t:.2f}')
# PoE: expert 1 informative, expert 2 noise -> weight near 1; symmetric experts -> weight near 0.5
n = 3000; yi = rng.integers(0, 10, n)
def expert(q):
    S = rng.normal(size=(n, 10)); S[np.arange(n), yi] += q; return log_softmax(S)
w, _ = fit_poe(expert(3.0), expert(0.0), yi); check('poe_prefers_informative', w >= 0.8, f'w={w}')
E1, E2 = expert(1.5), expert(1.5); w, _ = fit_poe(E1, E2, yi)
accs = (E1.argmax(1) == yi).mean(), (E2.argmax(1) == yi).mean(), ((0.5 * E1 + 0.5 * E2).argmax(1) == yi).mean()
check('poe_independent_experts_help', 0.25 <= w <= 0.75 and accs[2] > max(accs[:2]), f'w={w} accs={np.round(accs,3)}')
# epsilon mixture: exact against direct computation, and the bound holds
LA, LP = expert(1.0), expert(-5.0)
lm = eps_mixture_logp(LA, LP, 0.05)
check('mixture_exact', np.allclose(lm, np.log(0.05 * np.exp(LA) + 0.95 * np.exp(LP))))
idx = np.arange(n)
check('mixture_bound', np.all(-lm[idx, yi] <= -LA[idx, yi] - np.log(0.05) + 1e-12))
print('FAILS:', fails)
if fails: sys.exit(1)

In [ ]:
%%writefile test_analysis.py
import numpy as np, sys; sys.path.insert(0, '.')
from atlas_analysis import run_analysis
fails = []
def check(n, c, i=''):
    print('PASS' if c else 'FAIL', n, i)
    if not c: fails.append(n)

def synth(noise, seed=0, T=5, per=60, nbr=10, nct=10, M=200, dz=12, d=12, dr=40, buf_bias=0.0, sigma_from='reserve'):
    """Synthetic stream with known truth. buf_bias shifts the features of items that ever enter the buffer
    (a stand-in for 'the learner trained on these items'), so the matched bridge test has a known answer."""
    rng = np.random.default_rng(seed); C = T * 3
    tasks = list(rng.permutation(C).reshape(T, 3))
    mz = rng.normal(size=(C, dz)) * 1.2
    y = np.repeat(np.arange(C), per + nbr + nct); Z = mz[y] + rng.normal(size=(len(y), dz))
    yte = np.repeat(np.arange(C), 30); Zte = mz[yte] + rng.normal(size=(len(yte), dz))
    br, ctl, st = [], [], []
    for c in range(C):
        ids = rng.permutation(np.where(y == c)[0]); br += list(ids[:nbr]); ctl += list(ids[nbr:nbr + nct]); st += list(ids[nbr + nct:])
    st = np.array(st); bufs = []
    for t in range(T):
        seen = st[np.isin(y[st], np.concatenate(tasks[: t + 1]))]
        bufs.append(rng.choice(seen, min(M, len(seen)), replace=False))
    inbuf = np.zeros(len(y), bool); inbuf[np.concatenate(bufs)] = True
    W = rng.normal(size=(dz, d)); F = []; Ws = []
    for t in range(T):
        W = W + 0.4 * rng.normal(size=W.shape); Ws.append(W.copy())
        Ft = Z @ W + 0.3 * t + noise * rng.normal(size=(len(y), d))
        Ft[inbuf] += buf_bias * (t + 1) / T          # buffer items drift toward a common direction
        F.append(Ft.astype(np.float32))
    Wr = rng.normal(size=(dz, dr)); R = (np.tanh(Z @ Wr * 0.5) + 0.05 * rng.normal(size=(len(y), dr))).astype(np.float32)
    Rte = np.tanh(Zte @ Wr * 0.5); Fte = Zte @ Ws[-1] + 0.3 * (T - 1) + noise * rng.normal(size=(len(yte), d))
    return dict(tasks=tasks, y_train=y, stream_ids=st, bridge_ids=np.array(br, dtype=np.int64), control_ids=np.array(ctl),
                F=F, R=R, buffers=bufs, Ftest=Fte, Rtest=Rte, y_test=yte, head_acc=0.0, sigma_from=sigma_from)

if __name__ == '__main__':
    o = run_analysis(synth(0.0, nbr=40), 0)                     # exact linear features: transport recovers the oracle
    for k in ('CH_cv_buf', 'GR_prev_buf', 'GR_multi_buf', 'GR_prev_res_m', 'GR_prev_buf_m'):
        check('exact_' + k, abs(o['acc'][k] - o['acc']['O_full']) < 0.5, f"acc={o['acc'][k]:.1f} oracle={o['acc']['O_full']:.1f}")
    check('exact_hopR2', o['r2']['hop_min'] > 0.999)
    check('exact_no_fallbacks', sum(o['fallbacks'].values()) == 0, o['fallbacks'])
    check('bound_never_violated', sum(o['mixture_bound_violations'].values()) == 0)
    o = run_analysis(synth(1.0), 0)
    for k in sorted(o['acc'], key=lambda k: -o['acc'][k]): print(f"   {k:28s} acc={o['acc'][k]:.1f} nll={o['nll'].get(k, float('nan')):.3f}")
    check('noisy_greg_beats_bonly', o['acc']['GR_prev_buf'] >= o['acc']['B_only_buf'] - 0.5)
    check('poe_not_worse_than_parts', o['acc']['PoE[RC+GR_prev_buf]'] >= max(o['acc']['RC_LDA'], o['acc']['GR_prev_buf']) - 2.0)
    check('mixture_nll_bound_mean', o['nll']['Mix[RC,PoE+GR_prev_buf]'] <= o['nll']['RC_LDA'] - np.log(0.05) + 1e-9)
    check('bound_never_violated_noisy', sum(o['mixture_bound_violations'].values()) == 0)
    check('matched_n_close_to_cap', 8 < o['matched_n'] <= 10, o['matched_n'])
    # matched bridge with a known provenance bias: reserve must win; with no bias: no systematic winner
    gaps0, gaps1 = [], []
    for s in range(4):
        a = run_analysis(synth(0.5, seed=s, buf_bias=0.0), s); b = run_analysis(synth(0.5, seed=s, buf_bias=3.0), s)
        gaps0.append(a['acc']['GR_prev_res_m'] - a['acc']['GR_prev_buf_m']); gaps1.append(b['acc']['GR_prev_res_m'] - b['acc']['GR_prev_buf_m'])
    check('matched_detects_buffer_bias', np.mean(gaps1) > 3 and min(gaps1) > 0, np.round(gaps1, 1))
    check('matched_unbiased_no_gap', abs(np.mean(gaps0)) < 3, np.round(gaps0, 1))
    # equal-bytes config: no bridge, 2 control items per class, Sigma from buffer
    o = run_analysis(synth(1.0, nbr=0, nct=2, sigma_from='buffer'), 0)
    check('no_bridge_config_runs', 'GR_prev_res_m' not in o['acc'] and o['r2'] == {} and np.isfinite(list(o['acc'].values())).all())
    print('FAILS', fails)
    if fails: sys.exit(1)

In [ ]:
%%writefile sk_check.py
# ===== Cross-check our numpy code against scikit-learn =====
import numpy as np, sys; sys.path.insert(0, '.')
import atlas_lib
from sklearn.covariance import ledoit_wolf as sk_lw
from sklearn.linear_model import Ridge
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
rng = np.random.default_rng(7)
for n, p in ((300, 40), (30, 80)):
    X = rng.normal(size=(n, p)) @ rng.normal(size=(p, p)); Xc = X - X.mean(0)
    C1, s1 = atlas_lib.ledoit_wolf(Xc); C2, s2 = sk_lw(Xc, assume_centered=True)
    assert np.allclose(C1, C2, atol=1e-8 * np.abs(C2).max()) and abs(s1 - s2) < 1e-10, ('LW mismatch', n, p)
    Y = rng.normal(size=(n, 3)); lam = 2.5
    B1 = atlas_lib._ridge_factory(X, Y)(lam); B2 = Ridge(alpha=lam, fit_intercept=False, solver='svd').fit(X, Y).coef_.T
    assert np.allclose(B1, B2, atol=1e-6 * np.abs(B2).max()), ('ridge mismatch', n, p)
K, d = 7, 10
yl = np.repeat(np.arange(K), 60); Xl = rng.normal(size=(K, d))[yl] * 1.5 + rng.normal(size=(len(yl), d)) @ rng.normal(size=(d, d)) * 0.5
lda = LinearDiscriminantAnalysis(solver='lsqr', priors=np.ones(K) / K, store_covariance=True).fit(Xl, yl)
Xt = rng.normal(size=(2000, d)) * 2
agree = (atlas_lib.lda_predict(Xt, lda.means_, lda.covariance_) == lda.predict(Xt)).mean()
assert agree > 0.999, ('LDA mismatch', agree)
print('sklearn cross-check PASS: Ledoit-Wolf, ridge, LDA (agreement %.4f)' % agree)

In [ ]:
%%writefile audit_tests.py
# ===== Audit design tests (synthetic, known answers). Each checks a property ARCH-v2 relies on. =====
import numpy as np, sys; sys.path.insert(0, '.')
from atlas_lib import greg_means, global_affine_means, lda_predict, pooled_cov
rng = np.random.default_rng(123)
afails = []
def acheck(n, c, i=''):
    print('PASS' if c else 'FAIL', n, i)
    if not c: afails.append(n)

# U1 Re-anchoring does not reset inherited error: an exact hop carries the old error forward as delta @ W.
d, C = 8, 6
mu0 = {c: rng.normal(size=d) * 3 for c in range(C)}
W = rng.normal(size=(d, d)); b = rng.normal(size=d)
yb = np.repeat(np.arange(C), 30); A = np.stack([mu0[c] for c in yb]) + rng.normal(size=(len(yb), d)); Y = A @ W + b
delta = {c: rng.normal(size=d) * 0.5 for c in range(C)}
carried = {c: mu0[c] + delta[c] for c in range(C)}
est, _ = greg_means(list(range(C)), carried, A, Y, yb, rng)
err_after = np.array([np.linalg.norm(est[c] - (mu0[c] @ W + b)) for c in range(C)])
pred_err = np.array([np.linalg.norm(delta[c] @ W) for c in range(C)])
acheck('U1_inherited_error_persists', np.allclose(err_after, pred_err, rtol=1e-3) and err_after.min() > 0.1,
       f'after={err_after.round(3)} predicted={pred_err.round(3)}')

# U2 A rotation (no shift) is not fixed by a mean shift; a full linear transport fixes it.
Q, _ = np.linalg.qr(rng.normal(size=(d, d)))
Yr = A @ Q
shift = {c: carried_c + (Yr.mean(0) - A.mean(0)) for c, carried_c in mu0.items()}
full = global_affine_means(list(range(C)), mu0, A, Yr, yb, rng)
e_shift = max(np.linalg.norm(shift[c] - mu0[c] @ Q) for c in range(C)); e_full = max(np.linalg.norm(full[c] - mu0[c] @ Q) for c in range(C))
acheck('U2_rotation_needs_full_transport', e_shift > 1.0 and e_full < 1e-3, f'shift_err={e_shift:.2f} full_err={e_full:.1e}')

# U3 Reused evidence: fusing two estimates built from the SAME items as if independent halves the claimed
#    variance but not the real one. The design rule (never count reused evidence as new) is needed.
n, trials = 20, 4000
fused = []
for _ in range(trials):
    x = rng.normal(size=n); e1 = x.mean(); e2 = x.mean() + 0.0  # second "source" re-reads the same items
    fused.append((e1 + e2) / 2)
claimed = 1 / (n + n); real = np.var(fused)
acheck('U3_reused_evidence_overconfident', real / claimed > 1.8, f'real/claimed variance={real/claimed:.2f} (expect 2)')

# U4 A dormant class whose items fall inside active classes' support is still retrieved by the stable chart.
C2, dd = 50, 32
M = rng.normal(size=(C2, dd)) * 0.6       # crowded: classes overlap
Sig = np.eye(dd)
dormant = np.arange(10, 50)
Xq = np.repeat(dormant, 40); Zq = M[Xq] + rng.normal(size=(len(Xq), dd))
P = np.linalg.solve(Sig, M.T); S = Zq @ P - 0.5 * np.einsum('dc,cd->c', P, M)
top5 = np.argsort(-S, 1)[:, :5]; recall = (top5 == Xq[:, None]).any(1).mean()
acheck('U4_dormant_retrieved_top5', recall > 0.5, f'top-5 recall={recall:.2f} (chance 0.10)')

# U5 A label-only change (classes 0 and 1 swap labels) is caught by prequential error, not by input novelty.
C3 = 10; M3 = rng.normal(size=(C3, dd)) * 3
def novelty(Z): return np.min(((Z[:, None, :] - M3[None]) ** 2).sum(-1), 1)
lab = rng.integers(0, C3, 4000); Z = M3[lab] + rng.normal(size=(4000, dd))
before, after = slice(0, 2000), slice(2000, 4000)
obs = lab.copy(); sw = (np.arange(4000) >= 2000) & np.isin(lab, [0, 1]); obs[sw] = 1 - lab[sw]
pred = lda_predict(Z, M3, np.eye(dd))
err_b = (pred[before] != obs[before])[np.isin(obs[before], [0, 1])].mean(); err_a = (pred[after] != obs[after])[np.isin(obs[after], [0, 1])].mean()
nov = novelty(Z); dn = abs(nov[after].mean() - nov[before].mean()) / nov[before].std()
acheck('U5_label_shift_seen_by_error_not_novelty', err_a - err_b > 0.8 and dn < 0.1, f'err {err_b:.2f}->{err_a:.2f}, novelty shift={dn:.3f} sd')

# U6 The epsilon-mixture bound holds against a confidently wrong expert.
eps = 0.05; K = 20; N = 1000
y6 = rng.integers(0, K, N)
pA = rng.dirichlet(np.ones(K), N); pA[np.arange(N), y6] += 0.5; pA /= pA.sum(1, keepdims=True)
wrong = np.full((N, K), 1e-12); wrong[np.arange(N), (y6 + 1) % K] = 1; wrong /= wrong.sum(1, keepdims=True)
p = eps * pA + (1 - eps) * wrong
lhs = -np.log(p[np.arange(N), y6]); rhs = -np.log(pA[np.arange(N), y6]) - np.log(eps)
acheck('U6_eps_mixture_bound', np.all(lhs <= rhs + 1e-12) and -np.log(wrong[np.arange(N), y6]).mean() > 20,
       f'max(lhs-rhs)={np.max(lhs-rhs):.2e}, wrong-expert loss={-np.log(wrong[np.arange(N), y6]).mean():.1f}')
print('AUDIT FAILS:', afails)
if afails: sys.exit(1)

In [ ]:
def run_check(script):
    r = subprocess.run([sys.executable, script], capture_output=True, text=True)
    print(f'===== {script} =====\n' + r.stdout + (r.stderr[-3000:] if r.returncode else ''))
    if r.returncode != 0:
        raise SystemExit(f'{script} FAILED: stop here and send me this output.')
for s in ('test_lib.py', 'test_lib_v2.py', 'test_analysis.py', 'sk_check.py', 'audit_tests.py'):
    run_check(s)
print('All checks passed.')

In [ ]:
# GPU eigendecomposition for speed, used only if it gives the same ridge answers as numpy
def gpu_eigh(A):
    ev, V = torch.linalg.eigh(torch.from_numpy(np.ascontiguousarray(A)).to('cuda', torch.float64))
    return ev.cpu().numpy(), V.cpu().numpy()
atlas_lib.EIGH = np.linalg.eigh
if torch.cuda.is_available():
    rng = np.random.default_rng(0)
    X = rng.normal(size=(2000, 1152)); Y = rng.normal(size=(2000, 128))
    t = time.time(); B_cpu = atlas_lib.ridge_cv(X, Y, np.random.default_rng(1), split=1024)[0]; t_cpu = time.time() - t
    atlas_lib.EIGH = gpu_eigh
    t = time.time(); B_gpu = atlas_lib.ridge_cv(X, Y, np.random.default_rng(1), split=1024)[0]; t_gpu = time.time() - t
    same = np.allclose(B_cpu, B_gpu, rtol=1e-6, atol=1e-9 * np.abs(B_cpu).max())
    if not (same and t_gpu < t_cpu): atlas_lib.EIGH = np.linalg.eigh
    print(f'cpu {t_cpu:.1f}s, gpu {t_gpu:.1f}s, identical={same} -> using', 'GPU' if atlas_lib.EIGH is gpu_eigh else 'CPU')

## 3. Data and smoke tests (every code path, small and fast)

In [ ]:
data = load_cifar100()
SM = dict(n_classes=15, n_tasks=3, lr=0.03, momentum=0.9, batch=10, iters=2, aug=True, stream_per_class=60)
SMOKE_A = dict(SM, M=200, bridge_per_class=10, control_per_class=10, sigma_from='reserve')
SMOKE_E = dict(SM, M=180, bridge_per_class=0, control_per_class=2, sigma_from='buffer')
for name, cfg in (('A', SMOKE_A), ('E', SMOKE_E)):
    D = run_one(0, cfg, data)
    o = run_analysis(D, 0)
    assert all(np.isfinite(v) and 0 <= v <= 100 for v in o['acc'].values()), o['acc']
    assert sum(o['mixture_bound_violations'].values()) == 0
    assert ('GR_prev_res_m' in o['acc']) == (name == 'A')
    print(f'smoke {name} ok:', {k: round(v, 1) for k, v in o['acc'].items()})
    if name == 'A':
        D2 = run_one(0, cfg, data, log=lambda *a: None)
        dmax = float(np.abs(D['F'][-1] - D2['F'][-1]).max())
        print('determinism: max feature difference between identical runs =', dmax,
              '(OK)' if dmax < 1e-3 else '(WARNING: not bit-reproducible on this GPU; still valid as a 5-seed statistic)')
        del D2
    del D; gc.collect()
c = run_one(0, SMOKE_A, data, full=False); assert 0 <= c['oracle_ctl'] <= 100
print('smoke calibration path ok; random-chart control acc', round(rc_control_acc(0, SMOKE_A, data), 1))

## 4. Learner calibration (seed 100, control items only)

In [ ]:
BASE_A = dict(n_classes=100, n_tasks=20, batch=10, stream_per_class=None, M=2000, bridge_per_class=10, control_per_class=10, sigma_from='reserve')
CANDIDATES = {
    'L0_v1':        dict(lr=0.03, momentum=0.0, iters=1, aug=False),
    'L1_mom':       dict(lr=0.03, momentum=0.9, iters=1, aug=False),
    'L2_mom_it3':   dict(lr=0.01, momentum=0.9, iters=3, aug=True),
    'L3_mom_it3_hi': dict(lr=0.03, momentum=0.9, iters=3, aug=True),
    'L4_mom_it5':   dict(lr=0.01, momentum=0.9, iters=5, aug=True),
    'L5_sgd_it3':   dict(lr=0.1, momentum=0.0, iters=3, aug=True),
}
CAL_SEED = 100
CAL_PATH = 'atlas_v2b_calibration.json'
cal = json.load(open(CAL_PATH)) if os.path.exists(CAL_PATH) else {}
if cal.get('version') != VERSION: cal = {'version': VERSION, 'runs': {}}
if 'rc_ctl' not in cal:
    cal['rc_ctl'] = rc_control_acc(CAL_SEED, BASE_A, data)
print(f'random-chart LDA on control items: {cal["rc_ctl"]:.1f}  (the bar)')
for name, lc in CANDIDATES.items():
    if name in cal['runs']: print(name, 'done:', cal['runs'][name]); continue
    print(name, lc)
    try:
        cal['runs'][name] = run_one(CAL_SEED, dict(BASE_A, **lc), data, full=False)
    except Diverged as e:
        cal['runs'][name] = {'error': str(e)}; print('   diverged:', e)
    json.dump(cal, open(CAL_PATH, 'w'))
ok = {k: v for k, v in cal['runs'].items() if 'error' not in v}
assert ok, 'every learner candidate diverged: send me this output'
PICK = max(ok, key=lambda k: (ok[k]['oracle_ctl'], ok[k]['head_ctl']))
LEARNER = CANDIDATES[PICK]
GATE = ok[PICK]['oracle_ctl'] > cal['rc_ctl']
print(f'\nPICKED {PICK} {LEARNER}: learned ceiling {ok[PICK]["oracle_ctl"]:.1f} vs random chart {cal["rc_ctl"]:.1f} -> gate',
      'PASSED' if GATE else 'FAILED (learned-arm rankings will be provisional)')

## 5. The experiment (configs A and E × 5 seeds). Saved after every run.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
CONFIGS = {'A': dict(BASE_A, **LEARNER),
           'E': dict(BASE_A, **LEARNER, M=1800, bridge_per_class=0, control_per_class=2, sigma_from='buffer')}
RESULTS_PATH = 'atlas_v2b_partial.json'
results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
if results.get('_version') != VERSION or results.get('_learner') != PICK:
    results = {'_version': VERSION, '_learner': PICK}
t_all = time.time()
for cname, cfg in CONFIGS.items():
    results.setdefault(cname, {})
    for seed in SEEDS:
        if str(seed) in results[cname]: print(f'{cname} seed {seed}: done, skipping'); continue
        print(f'{cname} seed {seed}: running ...')
        try:
            D = run_one(seed, cfg, data)
        except Diverged as e:
            results[cname][str(seed)] = {'error': str(e)}; print('   diverged:', e)
            json.dump(results, open(RESULTS_PATH, 'w')); continue
        out = run_analysis(D, seed); out['config'] = cfg
        results[cname][str(seed)] = out
        del D; gc.collect()
        json.dump(results, open(RESULTS_PATH, 'w'))
        a = out['acc']
        print(f'   analysis {out["seconds"]:.0f}s | O_full {a["O_full"]:.1f} | RC {a["RC_LDA"]:.1f} | GR_prev_buf {a["GR_prev_buf"]:.1f}'
              f' | PoE {a["PoE[RC+GR_prev_buf]"]:.1f} | CH_cv_buf {a["CH_cv_buf"]:.1f} | head {a["ER_head"]:.1f} | {(time.time()-t_all)/60:.0f} min')

## 6. Results and pre-registered comparisons

In [ ]:
runs = {k: v for k, v in results.items() if not k.startswith('_')}
print(f'Learner: {PICK} {LEARNER} | gate {"PASSED" if GATE else "FAILED"} (learned ceiling {ok[PICK]["oracle_ctl"]:.1f} vs random chart {cal["rc_ctl"]:.1f} on control)')
print('Calibration:', {k: (v.get('oracle_ctl'), v.get('head_ctl')) if 'error' not in v else 'diverged' for k, v in cal['runs'].items()})
summary = summarize(runs)
final = dict(meta=dict(version=VERSION, torch=torch.__version__, numpy=np.__version__, gpu=GPU, seeds=SEEDS, configs=CONFIGS,
                       learner=PICK, gate=bool(GATE), calibration=cal, finished=time.strftime('%Y-%m-%d %H:%M:%S')),
             summary=summary, runs=runs)
json.dump(final, open('atlas_v2b_results.json', 'w'), indent=1)
try:
    from google.colab import files; files.download('atlas_v2b_results.json')
except Exception as e:
    print('Download atlas_v2b_results.json from the Files panel on the left.', e)